# Predict early-season FPL performance from the previous season aggregates


In [5]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split

from prediction.artifacts.io import save_trained_catboost_model
from prediction.artifacts.path_registry import PRE_SEASON_ARTIFACT_PATH
from training.load_training_data import load_historic_player_fixture_data
from training.config import RANDOM_STATE, EARLY_GAMEWEEKS, NUMERIC_FEATURES



# Training and evaluation

For training use aggregate values from the 2021/22 and 2022/23 seasons to predict the performance of the same player in the early gameweeks of the following season

## Load feature and target seasons


In [6]:
TRAINING_SEASONS = ["2021-22", "2022-23", "2023-24"]

fixture_history_df = pd.concat(
    [
        load_historic_player_fixture_data(season).assign(season=season)
        for season in TRAINING_SEASONS
    ],
    ignore_index=True,
    sort=False,
)

print(fixture_history_df.groupby("season").size())
fixture_history_df.head()

season
2021-22    25447
2022-23    26505
2023-24    29725
dtype: int64


,name,position,team,xP,assists,bonus,bps,clean_sheets,creativity,player_id,...,value,was_home,yellow_cards,GW,season,expected_assists,expected_goal_involvements,expected_goals,expected_goals_conceded,starts
0,Eric Bailly,DEF,Man Utd,0.0,0,0,0,0,0.0,286,...,50,True,0,1,2021-22,NaN,NaN,NaN,NaN,NaN
1,Keinan Davis,FWD,Aston Villa,0.4,0,0,0,0,0.0,49,...,45,False,0,1,2021-22,NaN,NaN,NaN,NaN,NaN
2,Ayotomiwa Dele-Bashiru,MID,Watford,0.0,0,0,0,0,0.0,394,...,45,True,0,1,2021-22,NaN,NaN,NaN,NaN,NaN
3,James Ward-Prowse,MID,Southampton,2.3,0,0,20,0,30.5,341,...,65,False,0,1,2021-22,NaN,NaN,NaN,NaN,NaN
4,Bruno Miguel Borges Fernandes,MID,Man Utd,4.4,0,3,61,0,35.9,277,...,120,True,0,1,2021-22,NaN,NaN,NaN,NaN,NaN


# Feature list

In [7]:
CATEGORICAL_COLUMNS = ["position"]

TARGET_COLUMN = "target_avg_fixture_points_first_gws"

# Create training dataset

In [8]:
fixture_history_df = fixture_history_df.sort_values(
    ["season", "name", "GW", "fixture_id"]
).copy()
fixture_history_df[NUMERIC_FEATURES] = fixture_history_df[
    NUMERIC_FEATURES
].apply(
    pd.to_numeric, errors="coerce"
)

player_season_categories = (
    fixture_history_df.groupby(["season", "name"], as_index=False)[
        CATEGORICAL_COLUMNS
    ].last()
)

player_season_totals = (
    fixture_history_df.groupby(["season", "name"], as_index=False)[
        NUMERIC_FEATURES
    ]
    .sum(min_count=1)
    .rename(columns={
        feature: f"season_sum_{feature}" for feature in NUMERIC_FEATURES
    })
)
player_season_features = player_season_categories.merge(
    player_season_totals,
    on=["season", "name"],
    how="inner",
    validate="one_to_one",
)

early_fixture_history = fixture_history_df.loc[
    fixture_history_df["GW"].between(1, EARLY_GAMEWEEKS)
].copy()

early_fixture_targets = (
    early_fixture_history.groupby(["season", "name"], as_index=False)
    .agg(**{TARGET_COLUMN: ("total_points", "mean")})
    .rename(columns={"season": "target_season"})
)

subsequent_season = dict(zip(TRAINING_SEASONS, TRAINING_SEASONS[1:]))
player_season_features["target_season"] = (
    player_season_features["season"].map(subsequent_season)
)

model_data = (
    player_season_features.dropna(subset=["target_season"])
    .merge(
        early_fixture_targets,
        on=["target_season", "name"],
        how="inner",
        validate="one_to_one",
    )
    .sort_values(["season", "name"])
    .reset_index(drop=True)
)

assert not model_data.duplicated(["season", "name"]).any()
print(model_data.groupby(["season", "target_season"]).size())
model_data.head()

season   target_season
2021-22  2022-23          399
2022-23  2023-24          493
dtype: int64


,season,name,position,season_sum_total_points,season_sum_minutes,season_sum_goals_scored,season_sum_assists,season_sum_clean_sheets,season_sum_goals_conceded,season_sum_own_goals,...,season_sum_creativity,season_sum_threat,season_sum_ict_index,season_sum_starts,season_sum_expected_goals,season_sum_expected_assists,season_sum_expected_goal_involvements,season_sum_expected_goals_conceded,target_season,target_avg_fixture_points_first_gws
0,2021-22,Aaron Cresswell,DEF,115,2726,2,4,7,38,0,...,542.4,151.0,127.0,NaN,NaN,NaN,NaN,NaN,2022-23,2.222222
1,2021-22,Aaron Ramsdale,GK,135,3060,0,0,12,39,0,...,1.0,0.0,68.0,NaN,NaN,NaN,NaN,NaN,2022-23,3.111111
2,2021-22,Aaron Wan-Bissaka,DEF,41,1793,0,0,3,35,0,...,237.0,123.0,73.3,NaN,NaN,NaN,NaN,NaN,2022-23,0.125000
3,2021-22,Abdoulaye Doucouré,MID,90,2536,2,5,5,47,0,...,285.7,441.0,120.6,NaN,NaN,NaN,NaN,NaN,2022-23,0.555556
4,2021-22,Adam Armstrong,FWD,57,1409,2,3,4,31,0,...,173.9,567.0,91.3,NaN,NaN,NaN,NaN,NaN,2022-23,2.777778


## Player-level train/validation split


In [9]:
model_features = CATEGORICAL_COLUMNS + [
    f"season_sum_{feature}" for feature in NUMERIC_FEATURES
]

train_index, valid_index = train_test_split(
    model_data.index, test_size=0.20, random_state=RANDOM_STATE
)
X_train = model_data.loc[train_index, model_features].copy()
X_valid = model_data.loc[valid_index, model_features].copy()
y_train = model_data.loc[train_index, TARGET_COLUMN].copy()
y_valid = model_data.loc[valid_index, TARGET_COLUMN].copy()
for frame in (X_train, X_valid):
    frame[CATEGORICAL_COLUMNS] = (
        frame[CATEGORICAL_COLUMNS].fillna("__MISSING__").astype(str)
    )

print(f"Train players: {len(X_train):,}; validation players: {len(X_valid):,}")


Train players: 713; validation players: 179


## Unweighted CatBoost model and dummy baseline


In [10]:
validation_predictions = {}

dummy_model = DummyRegressor(strategy="mean").fit(X_train, y_train)
validation_predictions["Dummy"] = dummy_model.predict(X_valid)

preseason_model = CatBoostRegressor(
    iterations=163,
    learning_rate=0.03,
    depth=6,
    loss_function="RMSE",
    random_seed=RANDOM_STATE,
    verbose=False,
    allow_writing_files=False,
)
preseason_model.fit(
    X_train,
    y_train,
    cat_features=CATEGORICAL_COLUMNS,
)
validation_predictions["Unweighted CatBoost"] = preseason_model.predict(X_valid)


## Comparison table


In [11]:
def evaluate_predictions(predictions, top_fraction=0.10):
    evaluation = pd.DataFrame({"actual": y_valid, "predicted": predictions})
    n = max(1, int(np.ceil(len(evaluation) * top_fraction)))
    predicted_top = evaluation.nlargest(n, "predicted")
    actual_top_indices = set(evaluation.nlargest(n, "actual").index)
    return {
        "MAE": mean_absolute_error(evaluation["actual"], evaluation["predicted"]),
        "RMSE": root_mean_squared_error(evaluation["actual"], evaluation["predicted"]),
        "R2": r2_score(evaluation["actual"], evaluation["predicted"]),
        "top_decile_avg_actual_points": predicted_top["actual"].mean(),
        "top_decile_hit_rate": predicted_top.index.isin(actual_top_indices).mean(),
        "top_decile_oracle_regret": (
            evaluation.nlargest(n, "actual")["actual"].mean()
            - predicted_top["actual"].mean()
        ),
    }

comparison_table = (
    pd.DataFrame.from_dict(
        {name: evaluate_predictions(preds) for name, preds in validation_predictions.items()},
        orient="index",
    )
    .rename_axis("model").reset_index()
    .sort_values(["top_decile_avg_actual_points", "MAE"], ascending=[False, True])
    .reset_index(drop=True)
)
comparison_table.round(3)


,model,MAE,RMSE,R2,top_decile_avg_actual_points,top_decile_hit_rate,top_decile_oracle_regret
0,Unweighted CatBoost,0.876,1.169,0.394,3.208,0.333,1.144
1,Dummy,1.314,1.506,-0.006,1.690,0.167,2.662


In [12]:
save_trained_catboost_model(
    model=preseason_model,
    feature_columns=model_features,
    categorical_columns=CATEGORICAL_COLUMNS,
    model_name="Unweighted CatBoost pre-season model",
    model_version="1.0",
    save_path=PRE_SEASON_ARTIFACT_PATH,
)
print(f"Saved pre-season model to {PRE_SEASON_ARTIFACT_PATH}")


Saved pre-season model to /Users/calumthompson/Documents/fantasy_football_v2/src/prediction/artifacts/trained_models/pre_season_model.joblib
